In [ ]:
# %%writefile 02_inference.ipynb
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Inference Notebook\n",
    "# Load and chat with your trained model"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# INSTALL DEPENDENCIES\n",
    "!pip install -q transformers accelerate bitsandbytes peft"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# IMPORTS\n",
    "import torch\n",
    "from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n",
    "from peft import PeftModel\n",
    "\n",
    "# CONFIG (edit these if needed)\n",
    "BASE_MODEL = \"codellama/CodeLlama-7b-Instruct-hf\"\n",
    "ADAPTER_PATH = \"./codellama-coding-chat-final\"  # Your trained LoRA\n",
    "USE_MERGED = False  # Set True if you already merged the model\n",
    "MERGED_PATH = \"./production-model-merged\"\n",
    "\n",
    "print(\"🚀 Loading model...\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# LOAD MODEL\n",
    "if USE_MERGED and os.path.exists(MERGED_PATH):\n",
    "    # Load merged model (faster)\n",
    "    print(\"Loading merged model...\")\n",
    "    tokenizer = AutoTokenizer.from_pretrained(MERGED_PATH)\n",
    "    model = AutoModelForCausalLM.from_pretrained(\n",
    "        MERGED_PATH,\n",
    "        torch_dtype=torch.float16,\n",
    "        device_map=\"auto\"\n",
    "    )\n",
    "else:\n",
    "    # Load base + LoRA adapter\n",
    "    print(\"Loading base model + LoRA adapter...\")\n",
    "    \n",
    "    bnb_config = BitsAndBytesConfig(\n",
    "        load_in_4bit=True,\n",
    "        bnb_4bit_quant_type=\"nf4\",\n",
    "        bnb_4bit_compute_dtype=torch.float16,\n",
    "        bnb_4bit_use_double_quant=True,\n",
    "    )\n",
    "    \n",
    "    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)\n",
    "    base_model = AutoModelForCausalLM.from_pretrained(\n",
    "        BASE_MODEL,\n",
    "        quantization_config=bnb_config,\n",
    "        device_map=\"auto\"\n",
    "    )\n",
    "    \n",
    "    # Load your trained adapter\n",
    "    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)\n",
    "    print(\"✅ LoRA adapter loaded!\")\n",
    "\n",
    "tokenizer.pad_token = tokenizer.eos_token\n",
    "model.eval()\n",
    "\n",
    "print(\"✅ Model ready for inference!\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# GENERATION FUNCTION\n",
    "def generate_response(prompt, max_tokens=512, temperature=0.7, top_p=0.9):\n",
    "    \"\"\"Generate response from model\"\"\"\n",
    "    formatted = f\"[INST] {prompt} [/INST]\"\n",
    "    \n",
    "    inputs = tokenizer(formatted, return_tensors=\"pt\").to(\"cuda\")\n",
    "    \n",
    "    with torch.no_grad():\n",
    "        outputs = model.generate(\n",
    "            **inputs,\n",
    "            max_new_tokens=max_tokens,\n",
    "            temperature=temperature,\n",
    "            top_p=top_p,\n",
    "            do_sample=True,\n",
    "            repetition_penalty=1.1,\n",
    "            pad_token_id=tokenizer.eos_token_id,\n",
    "        )\n",
    "    \n",
    "    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)\n",
    "    # Remove the prompt from response\n",
    "    response = full_response.replace(formatted, \"\").strip()\n",
    "    return response\n",
    "\n",
    "print(\"✅ Generation function ready!\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# TEST EXAMPLES\n",
    "test_prompts = [\n",
    "    \"Write a Python function to reverse a string\",\n",
    "    \"Explain what a neural network is\",\n",
    "    \"How do I handle errors in Python?\",\n",
    "    \"Write a SQL query to find duplicate emails\",\n",
    "    \"What is the difference between list and tuple in Python?\",\n",
    "]\n",
    "\n",
    "print(\"🧪 Running tests...\\n\")\n",
    "\n",
    "for i, prompt in enumerate(test_prompts, 1):\n",
    "    print(f\"Test {i}: {prompt}\")\n",
    "    response = generate_response(prompt, max_tokens=256)\n",
    "    print(f\"Response: {response[:300]}...\")\n",
    "    print(\"-\" * 50)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# INTERACTIVE CHAT\n",
    "print(\"🤖 Interactive Chat Mode (type 'exit' to quit)\")\n",
    "print(\"=\" * 50)\n",
    "\n",
    "while True:\n",
    "    user_input = input(\"\\nYou: \")\n",
    "    if user_input.lower() in ['exit', 'quit', 'bye']:\n",
    "        print(\"👋 Goodbye!\")\n",
    "        break\n",
    "    \n",
    "    print(\"AI: \", end=\"\", flush=True)\n",
    "    response = generate_response(user_input)\n",
    "    print(response)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}